# Phase 5: Time-Aware LSTM
## FreshCart AI — Temporal Context Embeddings

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import json
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {len(tf.config.list_physical_devices('GPU')) > 0}")

In [ ]:
drive.mount('/content/drive')

DATA_DIR   = '/content/drive/MyDrive/freshcart/data/processed'
MODELS_DIR = '/content/drive/MyDrive/freshcart/saved_models'

# Load item sequences (same npz files as Phase 4)
train_data = np.load(f'{DATA_DIR}/X_train.npz')
val_data   = np.load(f'{DATA_DIR}/X_val.npz')
test_data  = np.load(f'{DATA_DIR}/X_test.npz')

X_train = train_data['sequences']
X_val   = val_data['sequences']
X_test  = test_data['sequences']

# Load vocabulary
with open(f'{DATA_DIR}/vocab.json', 'r') as f:
    vocab = json.load(f)

VOCAB_SIZE = len(vocab)   # 5001 (5000 products + 1 PAD)
SEQ_LEN    = X_train.shape[1] - 1  # 99 (predict last token)

print(f"Train shape: {X_train.shape}")
print(f"Val shape:   {X_val.shape}")
print(f"Test shape:  {X_test.shape}")
print(f"Vocab size:  {VOCAB_SIZE}")

In [ ]:
# Sequence slicing: same leave-one-out protocol as Phase 3 & 4
X_train_seq = X_train[:, :-1]   # (N_train, 99) — context window
y_train     = X_train[:, -1]    # (N_train,)    — next-item target

X_val_seq   = X_val[:, :-1]     # (N_val, 99)
y_val       = X_val[:, -1]      # (N_val,)

X_test_seq  = X_test[:, :-1]    # (1000, 99)
y_test      = X_test[:, -1]     # (1000,)

print(f"X_train_seq: {X_train_seq.shape},  y_train: {y_train.shape}")
print(f"X_val_seq:   {X_val_seq.shape},    y_val:   {y_val.shape}")
print(f"X_test_seq:  {X_test_seq.shape},   y_test:  {y_test.shape}")

In [ ]:
# NOTE: In a full pipeline, these would come from the orders table joined to
# each user's last order before the test item. For now we generate realistic
# synthetic time features to enable architecture validation and training.
# Phase 8 (Backend) will wire real time context from user requests.

np.random.seed(42)

def make_time_features(n):
    hour     = np.random.randint(0, 24,  size=(n,)).astype(np.int32)   # 0–23
    dow      = np.random.randint(0, 7,   size=(n,)).astype(np.int32)   # 0–6
    days_gap = np.random.uniform(1, 30,  size=(n,)).astype(np.float32) / 30.0  # normalized 0–1 (D-02)
    return hour, dow, days_gap

hour_train, dow_train, days_train = make_time_features(len(X_train_seq))
hour_val,   dow_val,   days_val   = make_time_features(len(X_val_seq))
hour_test,  dow_test,  days_test  = make_time_features(len(X_test_seq))

print(f"hour_train shape: {hour_train.shape}, dtype: {hour_train.dtype}")
print(f"dow_train shape:  {dow_train.shape},  dtype: {dow_train.dtype}")
print(f"days_train shape: {days_train.shape}, dtype: {days_train.dtype}")
print(f"days_train range: [{days_train.min():.3f}, {days_train.max():.3f}]  (should be 0–1)")

In [ ]:
def build_time_aware_model(
    vocab_size=5001,
    seq_len=99,
    embed_dim=128,
    lstm_units=256,
    dropout_rate=0.3,
    hour_embed_dim=16,
    dow_embed_dim=8,
    days_dense_dim=8
):
    """
    Time-Aware LSTM Recommender — Keras Functional API.

    Inputs:
      seq_input     : (batch, 99)  int32   — item sequence
      hour_input    : (batch,)     int32   — hour of day (0-23)
      dow_input     : (batch,)     int32   — day of week (0-6)
      days_gap_input: (batch,)     float32 — days since prior order (normalized 0-1)

    Architecture:
      Sequence path:  Embedding(5001,128,mask_zero=True) → LSTM(256,ret_seq=True) → LSTM(256) → Dropout(0.3)
      Time path:      Embed(24→16) + Embed(7→8) + Dense(1→8) → concat → 32-dim time vector
      Fusion:         concat([256-dim LSTM, 32-dim time]) → 288-dim → Dense(5001, softmax)
    """
    # --- Inputs ---
    seq_input = keras.Input(shape=(seq_len,), dtype='int32', name='seq_input')
    hour_input = keras.Input(shape=(), dtype='int32', name='hour_input')
    dow_input = keras.Input(shape=(), dtype='int32', name='dow_input')
    days_gap_input = keras.Input(shape=(1,), dtype='float32', name='days_gap_input')

    # --- Sequence path ---
    x = keras.layers.Embedding(
        input_dim=vocab_size,
        output_dim=embed_dim,
        mask_zero=True,            # D-04: propagate padding mask through LSTMs
        name='item_embedding'
    )(seq_input)
    x = keras.layers.LSTM(lstm_units, return_sequences=True, name='lstm_1')(x)
    x = keras.layers.LSTM(lstm_units, return_sequences=False, name='lstm_2')(x)
    x = keras.layers.Dropout(dropout_rate, name='dropout')(x)
    # x shape: (batch, 256)

    # Time feature: hour-of-day embedding (24 → 16)
    h = keras.layers.Embedding(24, hour_embed_dim, name='hour_embedding')(hour_input)
    h = keras.layers.Flatten(name='hour_flat')(h)
    # h shape: (batch, 16)

    # Time feature: day-of-week embedding (7 → 8)
    d = keras.layers.Embedding(7, dow_embed_dim, name='dow_embedding')(dow_input)
    d = keras.layers.Flatten(name='dow_flat')(d)
    # d shape: (batch, 8)

    # Time feature: days-gap scalar → Dense 8 (D-02: already normalized 0-1)
    g = keras.layers.Dense(days_dense_dim, activation='relu', name='days_gap_dense')(days_gap_input)
    # g shape: (batch, 8)

    # Concatenate all time features → 32-dim time vector
    time_vec = keras.layers.Concatenate(name='time_concat')([h, d, g])
    # time_vec shape: (batch, 32)

    # Fuse LSTM output (256) + time vector (32) → 288-dim
    fused = keras.layers.Concatenate(name='fusion_concat')([x, time_vec])
    # fused shape: (batch, 288)

    # Output projection
    output = keras.layers.Dense(vocab_size, activation='softmax', name='output_layer')(fused)

    model = keras.Model(
        inputs=[seq_input, hour_input, dow_input, days_gap_input],
        outputs=output,
        name='TimeAwareLSTM'
    )
    return model

model = build_time_aware_model()
model.summary()

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
print("TimeAwareLSTM compiled. Ready for training.")

In [ ]:
EPOCHS     = 20
BATCH_SIZE = 512   # D-07: same as Phase 4

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=f'{MODELS_DIR}/lstm_time_model.h5',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
]

print(f"Training config: epochs={EPOCHS}, batch_size={BATCH_SIZE}")
print(f"Model checkpoint: {MODELS_DIR}/lstm_time_model.h5")

In [ ]:
# D-05: pass numpy arrays directly to model.fit()
# 4-input model requires matching dict or list of arrays
history = model.fit(
    x={
        'seq_input':      X_train_seq,
        'hour_input':     hour_train,
        'dow_input':      dow_train,
        'days_gap_input': days_train.reshape(-1, 1)   # shape (N, 1)
    },
    y=y_train,
    validation_data=(
        {
            'seq_input':      X_val_seq,
            'hour_input':     hour_val,
            'dow_input':      dow_val,
            'days_gap_input': days_val.reshape(-1, 1)
        },
        y_val
    ),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1
)

print(f"
Training complete. Best val_loss epoch saved to lstm_time_model.h5")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss curves
axes[0].plot(history.history['loss'],     label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_title('TimeAwareLSTM — Training & Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy curves
axes[1].plot(history.history['accuracy'],     label='Train Accuracy')
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[1].set_title('TimeAwareLSTM — Training & Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/lstm_time_training_curves.png', dpi=150)
plt.show()
print("Training curves saved.")

In [ ]:
# Verify the checkpoint was saved and reloads correctly
import os

model_path = f'{MODELS_DIR}/lstm_time_model.h5'
assert os.path.exists(model_path), f"Model not found at {model_path}"

# Reload and confirm it accepts 4-input inference
loaded_model = keras.models.load_model(model_path)
print(f"Model loaded from: {model_path}")
print(f"Input names: {[inp.name for inp in loaded_model.inputs]}")

# Quick sanity inference (1 sample)
dummy_seq  = np.zeros((1, 99), dtype=np.int32)
dummy_hour = np.array([14], dtype=np.int32)
dummy_dow  = np.array([2],  dtype=np.int32)
dummy_days = np.array([[0.5]], dtype=np.float32)

preds = loaded_model.predict({
    'seq_input':      dummy_seq,
    'hour_input':     dummy_hour,
    'dow_input':      dummy_dow,
    'days_gap_input': dummy_days
}, verbose=0)

print(f"Prediction shape: {preds.shape}")   # should be (1, 5001)
print(f"Prediction sum:   {preds.sum():.4f}")  # should be ~1.0 (softmax)
print("Model verified. lstm_time_model.h5 is valid.")